In [1]:
## 1. Cleaning: bronze (Parquet) -> silver (clean Parquet)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum, avg, max, count, date_format, to_timestamp

spark = SparkSession.builder \
    .appName("NYC Subway - Part 1 - clean") \
# Spark has only 1 GB so we increase to 4GB of memory to avoid crash knowing the size of data 
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

BRONZE = "/home/jovyan/project/data/bronze"
SILVER = "/home/jovyan/project/data/silver"
RESULTS = "/home/jovyan/project/results"

In [3]:
## 2. Ridership  1st Rule R1: keep the subway only (problem #4)
rides = spark.read.parquet(BRONZE + "/ridership")
r0 = rides.count()
rides = rides.where("transit_mode = 'subway'")
r1 = rides.count()
print("R1:", r0, "->", r1)

R1: 29375687 -> 29145131


In [4]:
## 3. Ridership 2nd Rule R2: missing or impossible values to drop empty or negative values 
rides = rides.where("transit_timestamp is not null and station_complex_id is not null and ridership >= 0")
r2 = rides.count()
print("R2:", r1, "->", r2)

R2: 29145131 -> 29145131


In [5]:
## 4. Ridership 3rd rule R3: keep the columns we need, with the right type that means that we keep 5 columns and fix their types (station ID is a text → switched to a number, ridership is a decimal nnumber  → convert into a whole number)
rides = rides.select(
    col("transit_timestamp"),
    col("station_complex_id").cast("int").alias("complex_id"),
    col("payment_method"),
    col("fare_class_category"),
    col("ridership").cast("int").alias("ridership"),
)
rides = rides.where("complex_id is not null")
r3 = rides.count()
print("R3:", r2, "->", r3)

R3: 29145131 -> 29145131


In [6]:
## 5. Ridership  4th rule R4: the hour that does not exist in New York
# 9 March 2025: clocks jump from 01:59 to 03:00, so 02:00-02:59 never happened so we remve that hour that never existed
rides = rides.where("transit_timestamp != '2025-03-09 02:00:00'")
r4 = rides.count()
print("R4:", r3, "->", r4)

R4: 29145131 -> 29143881


In [8]:
## 6. Ridership 5th rule  R5: We delete the duplicate rows
rides = rides.distinct()
r5 = rides.count()
print("R5:", r4, "->", r5)

R5: 29143881 -> 29143881
